# Grok-ml-t4x2-smoke

> **Domain:** ml · **Task:** t4x2-smoke  
> **Machine:** Kaggle `NvidiaTeslaT4` (= **T4×2** community accelerator)

## Goals
1. Confirm **both** T4 GPUs are visible (`torch.cuda.device_count() == 2`)
2. FP32 GEMM micro-benchmark on `cuda:0`
3. Tiny multi-GPU DataParallel CNN train (synthetic data — no downloads)
4. Write `results.json` under `/kaggle/working` for CLI download

## Naming
`Grok-{domain}-{task}` → **Grok-ml-t4x2-smoke**


In [ ]:
# -*- setup: both T4s visible -*-
import os, json, time, platform, traceback
from pathlib import Path

# Do NOT pin CUDA_VISIBLE_DEVICES — we want T4×2
os.environ.pop("CUDA_VISIBLE_DEVICES", None)
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

OUT = Path("/kaggle/working")
OUT.mkdir(parents=True, exist_ok=True)

print("torch", torch.__version__)
print("python", platform.python_version())
print("cuda_available", torch.cuda.is_available())
assert torch.cuda.is_available(), "CUDA not available — accelerator not attached"
n_gpu = torch.cuda.device_count()
print("device_count", n_gpu)
for i in range(n_gpu):
    p = torch.cuda.get_device_properties(i)
    print(
        f"device[{i}]", torch.cuda.get_device_name(i),
        "cap", torch.cuda.get_device_capability(i),
        "mem_gb", round(p.total_memory / 1e9, 2),
    )

# Soft warn if only 1 GPU (session may still be single-T4); hard-require >=1
assert n_gpu >= 1, "No CUDA devices"
if n_gpu < 2:
    print("WARNING: expected T4×2 (2 devices); continuing with", n_gpu)
else:
    print("OK: dual-GPU session detected (T4×2)")

device = torch.device("cuda:0")
print("primary", device)


In [ ]:
# -*- FP32 GEMM micro-benchmark -*-
N = 4096
a = torch.randn(N, N, device=device)
b = torch.randn(N, N, device=device)
for _ in range(3):
    c = a @ b
torch.cuda.synchronize()
t0 = time.perf_counter()
iters = 10
for _ in range(iters):
    c = a @ b
torch.cuda.synchronize()
elapsed = time.perf_counter() - t0
flops = 2 * (N ** 3) * iters
tflops = flops / elapsed / 1e12
print(f"GEMM {N}x{N} x{iters}: {elapsed:.3f}s, ~{tflops:.2f} TFLOPS (FP32)")
gemm = {"n": N, "iters": iters, "seconds": round(elapsed, 4), "tflops_fp32": round(tflops, 3)}


In [ ]:
# -*- tiny CNN + DataParallel if multi-GPU -*-
class TinyCNN(nn.Module):
    def __init__(self, n_classes=10):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(inplace=True), nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(128, n_classes),
        )

    def forward(self, x):
        return self.net(x)

torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

n_train, n_val = 4096, 512
x_train = torch.randn(n_train, 3, 32, 32)
y_train = torch.randint(0, 10, (n_train,))
x_val = torch.randn(n_val, 3, 32, 32)
y_val = torch.randint(0, 10, (n_val,))

train_loader = DataLoader(TensorDataset(x_train, y_train), batch_size=128, shuffle=True, num_workers=0)
val_loader = DataLoader(TensorDataset(x_val, y_val), batch_size=256, shuffle=False, num_workers=0)

raw_model = TinyCNN().to(device)
if torch.cuda.device_count() > 1:
    model = nn.DataParallel(raw_model)
    print("wrapped with DataParallel on", torch.cuda.device_count(), "GPUs")
else:
    model = raw_model
    print("single-GPU training")

opt = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

history = []
epochs = 5
t0 = time.perf_counter()
for epoch in range(1, epochs + 1):
    model.train()
    total_loss, n = 0.0, 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad(set_to_none=True)
        logits = model(xb)
        loss = loss_fn(logits, yb)
        loss.backward()
        opt.step()
        total_loss += loss.item() * xb.size(0)
        n += xb.size(0)
    train_loss = total_loss / max(n, 1)

    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            pred = model(xb).argmax(dim=1)
            correct += (pred == yb).sum().item()
            total += yb.size(0)
    val_acc = correct / max(total, 1)
    history.append({"epoch": epoch, "train_loss": round(train_loss, 5), "val_acc": round(val_acc, 5)})
    print(f"epoch {epoch}/{epochs} train_loss={train_loss:.4f} val_acc={val_acc:.4f}")

train_seconds = round(time.perf_counter() - t0, 3)
print("train_seconds", train_seconds)
assert history[-1]["train_loss"] < history[0]["train_loss"] or history[-1]["val_acc"] >= 0.08, (
    "Training did not make progress — check GPU kernels"
)


In [ ]:
# -*- persist results -*-
n_gpu = torch.cuda.device_count()
out = {
    "notebook": "Grok-ml-t4x2-smoke",
    "domain": "ml",
    "task": "t4x2-smoke",
    "ok": True,
    "cuda_available": True,
    "device_count": n_gpu,
    "device_names": [torch.cuda.get_device_name(i) for i in range(n_gpu)],
    "dual_t4": n_gpu >= 2,
    "gemm": gemm,
    "train_seconds": train_seconds,
    "history": history,
    "params": sum(p.numel() for p in raw_model.parameters()),
    "data_parallel": n_gpu > 1,
}
path = OUT / "results.json"
path.write_text(json.dumps(out, indent=2))
print("wrote", path)
print(json.dumps(out, indent=2))

# marker file for CLI automation
(OUT / "SUCCESS").write_text("ok\n")
torch.save({"history": history}, OUT / "tiny_cnn_history.pt")
print("SUCCESS")
